In [52]:
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

In [53]:
from pathlib import Path
import pandas as pd
data_path = Path("../data/employee_attrition_selected.csv")

if not data_path.exists():
    data_path = Path("backend/data/employee_attrition_selected.csv")

df = pd.read_csv(data_path)

print("Dataset shape:", df.shape)
df.head()

Dataset shape: (1470, 33)


,Age,BusinessTravel,Department,DistanceFromHome,Education,EducationField,EnvironmentSatisfaction,Gender,JobInvolvement,JobLevel,...,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager,CompanyExperienceRatio,PromotionFrequency,IncomePerYearExperience,SatisfactionScore,RoleTenureRatio,Attrition
0,41,Travel_Rarely,Sales,1,2,Life Sciences,2,Female,3,2,...,6,4,0,5,0.750000,0.000,749.125000,2.00,0.666667,Yes
1,49,Travel_Frequently,Research & Development,8,1,Life Sciences,3,Male,2,2,...,10,7,1,7,1.000000,0.100,513.000000,3.00,0.700000,No
2,37,Travel_Rarely,Research & Development,2,2,Other,4,Male,2,1,...,0,0,0,0,0.000000,0.000,298.571429,3.00,0.000000,Yes
3,33,Travel_Frequently,Research & Development,3,4,Life Sciences,4,Female,3,1,...,8,7,3,0,1.000000,0.375,363.625000,3.25,0.875000,No
4,27,Travel_Rarely,Research & Development,2,1,Medical,1,Male,3,1,...,2,2,2,2,0.333333,1.000,578.000000,2.50,1.000000,No


In [54]:
print("Columns:")
print(df.columns.tolist())

print("\nTotal columns:", len(df.columns))

Columns:
['Age', 'BusinessTravel', 'Department', 'DistanceFromHome', 'Education', 'EducationField', 'EnvironmentSatisfaction', 'Gender', 'JobInvolvement', 'JobLevel', 'JobRole', 'JobSatisfaction', 'MaritalStatus', 'MonthlyIncome', 'NumCompaniesWorked', 'OverTime', 'PercentSalaryHike', 'PerformanceRating', 'RelationshipSatisfaction', 'StockOptionLevel', 'TotalWorkingYears', 'TrainingTimesLastYear', 'WorkLifeBalance', 'YearsAtCompany', 'YearsInCurrentRole', 'YearsSinceLastPromotion', 'YearsWithCurrManager', 'CompanyExperienceRatio', 'PromotionFrequency', 'IncomePerYearExperience', 'SatisfactionScore', 'RoleTenureRatio', 'Attrition']

Total columns: 33


Separate Features and Target

In [55]:
X = df.drop("Attrition", axis=1)
y = df["Attrition"].map({"Yes": 1, "No": 0})

print("Number of features:", X.shape[1])
print("Target distribution:")
print(y.value_counts())

Number of features: 32
Target distribution:
Attrition
0    1233
1     237
Name: count, dtype: int64


Identify Categorical and Numerical Features

In [56]:
categorical_features = X.select_dtypes(include=["object"]).columns.tolist()
numeric_features = X.select_dtypes(exclude=["object"]).columns.tolist()

print("Categorical features:")
print(categorical_features)

print("\nNumerical features:")
print(numeric_features)

Categorical features:
['BusinessTravel', 'Department', 'EducationField', 'Gender', 'JobRole', 'MaritalStatus', 'OverTime']

Numerical features:
['Age', 'DistanceFromHome', 'Education', 'EnvironmentSatisfaction', 'JobInvolvement', 'JobLevel', 'JobSatisfaction', 'MonthlyIncome', 'NumCompaniesWorked', 'PercentSalaryHike', 'PerformanceRating', 'RelationshipSatisfaction', 'StockOptionLevel', 'TotalWorkingYears', 'TrainingTimesLastYear', 'WorkLifeBalance', 'YearsAtCompany', 'YearsInCurrentRole', 'YearsSinceLastPromotion', 'YearsWithCurrManager', 'CompanyExperienceRatio', 'PromotionFrequency', 'IncomePerYearExperience', 'SatisfactionScore', 'RoleTenureRatio']


Train-Test Split

In [57]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

Training samples: 1176
Testing samples: 294


Preprocessing Pipeline

Categorical columns → One-Hot Encoding, 
Numerical columns → Standard Scaling

In [58]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            StandardScaler(),
            numeric_features
        ),
        (
            "categorical",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            ),
            categorical_features
        )
    ]
)

print("Preprocessing pipeline created successfully!")

Preprocessing pipeline created successfully!


# Model 1 — Logistic Regression

In [59]:
logistic_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "model",
            LogisticRegression(
                max_iter=1000,
                class_weight="balanced",
                random_state=42
            )
        )
    ]
)

logistic_model.fit(X_train, y_train)

print("Logistic Regression trained successfully!")

Logistic Regression trained successfully!


In [61]:
y_pred_lr = logistic_model.predict(X_test)
y_prob_lr = logistic_model.predict_proba(X_test)[:, 1]

print("Logistic Regression")
print("-------------------------")
print("Accuracy :", round(accuracy_score(y_test, y_pred_lr), 4))
print("Precision:", round(precision_score(y_test, y_pred_lr), 4))
print("Recall   :", round(recall_score(y_test, y_pred_lr), 4))
print("F1 Score :", round(f1_score(y_test, y_pred_lr), 4))
print("ROC-AUC  :", round(roc_auc_score(y_test, y_prob_lr), 4))

Logistic Regression
-------------------------
Accuracy : 0.7721
Precision: 0.3864
Recall   : 0.7234
F1 Score : 0.5037
ROC-AUC  : 0.8188


# Model 2 — Random Forest

In [62]:
random_forest_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "model",
            RandomForestClassifier(
                n_estimators=300,
                class_weight="balanced",
                random_state=42
            )
        )
    ]
)

random_forest_model.fit(X_train, y_train)

print("Random Forest trained successfully!")

Random Forest trained successfully!


In [63]:
y_pred_rf = random_forest_model.predict(X_test)
y_prob_rf = random_forest_model.predict_proba(X_test)[:, 1]

print("Random Forest")
print("-------------------------")
print("Accuracy :", round(accuracy_score(y_test, y_pred_rf), 4))
print("Precision:", round(precision_score(y_test, y_pred_rf), 4))
print("Recall   :", round(recall_score(y_test, y_pred_rf), 4))
print("F1 Score :", round(f1_score(y_test, y_pred_rf), 4))
print("ROC-AUC  :", round(roc_auc_score(y_test, y_prob_rf), 4))

Random Forest
-------------------------
Accuracy : 0.8435
Precision: 0.5556
Recall   : 0.1064
F1 Score : 0.1786
ROC-AUC  : 0.8105


Compare Both Models

In [64]:
results = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Random Forest"
    ],
    "Accuracy": [
        accuracy_score(y_test, y_pred_lr),
        accuracy_score(y_test, y_pred_rf)
    ],
    "Precision": [
        precision_score(y_test, y_pred_lr),
        precision_score(y_test, y_pred_rf)
    ],
    "Recall": [
        recall_score(y_test, y_pred_lr),
        recall_score(y_test, y_pred_rf)
    ],
    "F1 Score": [
        f1_score(y_test, y_pred_lr),
        f1_score(y_test, y_pred_rf)
    ],
    "ROC-AUC": [
        roc_auc_score(y_test, y_prob_lr),
        roc_auc_score(y_test, y_prob_rf)
    ]
})

results.round(4)

,Model,Accuracy,Precision,Recall,F1 Score,ROC-AUC
0,Logistic Regression,0.7721,0.3864,0.7234,0.5037,0.8188
1,Random Forest,0.8435,0.5556,0.1064,0.1786,0.8105


Confusion Matrix

In [65]:
cm = confusion_matrix(y_test, y_pred_lr)

print("Logistic Regression Confusion Matrix:")
print(cm)

Logistic Regression Confusion Matrix:
[[193  54]
 [ 13  34]]


Classification Report

In [66]:
print(classification_report(
    y_test,
    y_pred_lr,
    target_names=["No Attrition", "Attrition"]
))

              precision    recall  f1-score   support

No Attrition       0.94      0.78      0.85       247
   Attrition       0.39      0.72      0.50        47

    accuracy                           0.77       294
   macro avg       0.66      0.75      0.68       294
weighted avg       0.85      0.77      0.80       294



In [67]:
# Compare models based on F1 Score
model_scores = {
    "Logistic Regression": f1_score(y_test, y_pred_lr),
    "Random Forest": f1_score(y_test, y_pred_rf)
}

best_model_name = max(model_scores, key=model_scores.get)
best_score = model_scores[best_model_name]

models = {
    "Logistic Regression": logistic_model,
    "Random Forest": random_forest_model
}

best_model = models[best_model_name]

print("Model Selection Results:")
for model_name, score in model_scores.items():
    print(f"{model_name}: F1 Score = {score:.4f}")

print("\nBest Model:", best_model_name)
print("Best F1 Score:", round(best_score, 4))

Model Selection Results:
Logistic Regression: F1 Score = 0.5037
Random Forest: F1 Score = 0.1786

Best Model: Logistic Regression
Best F1 Score: 0.5037


In [68]:
model_path = Path("../models/attrition_model.pkl")

model_path.parent.mkdir(parents=True, exist_ok=True)

joblib.dump(best_model, model_path)

print(f"Best model ({best_model_name}) saved successfully!")
print("Path:", model_path)

Best model (Logistic Regression) saved successfully!
Path: ..\models\attrition_model.pkl


In [69]:
import sklearn
print(sklearn.__version__)

1.6.0
